# 03 — Six Agentic AI Design Patterns for MarTech

**Chapter 2 | Mastering Agentic AI for Marketing Technology**

This notebook demonstrates all six foundational agentic design patterns,
each illustrated with a real-world marketing technology example:

1. **Reflection** — Self-critique and improve ad copy
2. **Tool Use** — CRM lookup during conversation
3. **Planning** — Multi-step campaign planning
4. **Multi-Agent** — Specialised agent collaboration
5. **Memory** — Conversation history for personalisation
6. **Guardrails** — Brand safety validation

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

USE_MOCK = os.getenv("USE_MOCK_APIS", "true").lower() == "true"

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY", "mock-key"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1")

def mock_completion(content: str) -> str:
    """Return mock response when USE_MOCK is True."""
    return content

def chat(messages: list[dict], **kwargs) -> str:
    """Unified chat helper with mock fallback."""
    if USE_MOCK:
        return mock_completion(f"[Mock response for: {messages[-1]['content'][:80]}...]")
    resp = client.chat.completions.create(model=MODEL, messages=messages, **kwargs)
    return resp.choices[0].message.content

print(f"Mock mode: {USE_MOCK}")

## Pattern 1: Reflection

The agent generates output, then critiques and improves its own work.
In MarTech: generate ad copy → self-critique for conversion best practices → refine.

In [ ]:
def reflection_pattern(product: str, audience: str, iterations: int = 2) -> dict:
    """Generate and iteratively refine ad copy using self-reflection."""
    # Step 1: Initial generation
    gen_messages = [
        {"role": "system", "content": "You are an expert performance marketer. Write concise, high-converting ad copy."},
        {"role": "user", "content": f"Write a Google Ads headline + description for {product} targeting {audience}."}
    ]
    draft = chat(gen_messages)
    history = [{"iteration": 0, "type": "draft", "content": draft}]

    for i in range(1, iterations + 1):
        # Step 2: Critique
        critique_messages = [
            {"role": "system", "content": (
                "You are a conversion rate optimisation expert. Critique the ad copy below. "
                "Check: emotional triggers, urgency, clarity, CTA strength, character limits."
            )},
            {"role": "user", "content": f"Critique this ad copy:\n{draft}"}
        ]
        critique = chat(critique_messages)
        history.append({"iteration": i, "type": "critique", "content": critique})

        # Step 3: Refine based on critique
        refine_messages = [
            {"role": "system", "content": "You are an expert copywriter. Improve the ad copy based on the critique."},
            {"role": "user", "content": f"Original:\n{draft}\n\nCritique:\n{critique}\n\nWrite an improved version."}
        ]
        draft = chat(refine_messages)
        history.append({"iteration": i, "type": "refined", "content": draft})

    return {"final_copy": draft, "history": history}

# Demo
result = reflection_pattern("AI-powered CRM platform", "B2B SaaS marketers")
for step in result["history"]:
    print(f"\n--- Iteration {step['iteration']} ({step['type']}) ---")
    print(step["content"])

## Pattern 2: Tool Use

The agent can call external tools (APIs, databases, functions) during reasoning.
In MarTech: look up a contact in the CRM to personalise a response.

In [ ]:
# Define CRM tool
CRM_DATABASE = {
    "alice@techcorp.com": {"name": "Alice Chen", "company": "TechCorp", "plan": "Enterprise", "mrr": 4500, "health": "green"},
    "bob@startup.io": {"name": "Bob Martinez", "company": "StartupIO", "plan": "Growth", "mrr": 800, "health": "yellow"},
    "carol@bigco.com": {"name": "Carol Williams", "company": "BigCo", "plan": "Enterprise", "mrr": 12000, "health": "red"},
}

def crm_lookup(email: str) -> str:
    """Look up a contact in the CRM by email."""
    contact = CRM_DATABASE.get(email)
    if contact:
        return json.dumps(contact)
    return json.dumps({"error": f"No contact found for {email}"})

# Define tool schema for OpenAI function calling
tools = [{
    "type": "function",
    "function": {
        "name": "crm_lookup",
        "description": "Look up a contact in the CRM by their email address.",
        "parameters": {
            "type": "object",
            "properties": {"email": {"type": "string", "description": "Contact email address"}},
            "required": ["email"]
        }
    }
}]

def tool_use_pattern(query: str) -> str:
    """Run an agent that can look up CRM data via tool calling."""
    messages = [
        {"role": "system", "content": "You are a customer success assistant. Use tools to look up customer info."},
        {"role": "user", "content": query}
    ]

    if USE_MOCK:
        # Simulate tool call flow
        email = "alice@techcorp.com" if "alice" in query.lower() else "bob@startup.io"
        result = crm_lookup(email)
        return f"[Mock] Looked up {email}: {result}"

    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    msg = response.choices[0].message

    if msg.tool_calls:
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = crm_lookup(args["email"])
            messages.append(msg)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        final = client.chat.completions.create(model=MODEL, messages=messages)
        return final.choices[0].message.content
    return msg.content

# Demo
print(tool_use_pattern("What's Alice's current plan and account health?"))

## Pattern 3: Planning

The agent decomposes a complex goal into a sequence of steps before execution.
In MarTech: break down "launch a product campaign" into actionable steps.

In [ ]:
def planning_pattern(goal: str) -> dict:
    """Decompose a marketing goal into a structured plan."""
    plan_messages = [
        {"role": "system", "content": (
            "You are a marketing operations strategist. Given a goal, create a detailed "
            "execution plan as a JSON array of steps. Each step has: id, action, channel, "
            "dependencies (list of step ids), and estimated_hours."
        )},
        {"role": "user", "content": f"Create an execution plan for: {goal}"}
    ]

    if USE_MOCK:
        plan = [
            {"id": 1, "action": "Define target audience segments in CDP", "channel": "Segment", "dependencies": [], "estimated_hours": 4},
            {"id": 2, "action": "Create email nurture sequence", "channel": "Klaviyo", "dependencies": [1], "estimated_hours": 6},
            {"id": 3, "action": "Design landing page", "channel": "CMS", "dependencies": [1], "estimated_hours": 8},
            {"id": 4, "action": "Set up paid search campaigns", "channel": "Google Ads", "dependencies": [3], "estimated_hours": 4},
            {"id": 5, "action": "Configure HubSpot workflows", "channel": "HubSpot", "dependencies": [2, 3], "estimated_hours": 3},
            {"id": 6, "action": "Launch and monitor", "channel": "All", "dependencies": [4, 5], "estimated_hours": 2}
        ]
    else:
        raw = chat(plan_messages)
        plan = json.loads(raw)

    return {"goal": goal, "steps": plan, "total_hours": sum(s["estimated_hours"] for s in plan)}

# Demo
result = planning_pattern("Launch a product-led growth campaign for our new analytics feature")
print(f"Goal: {result['goal']}")
print(f"Total estimated hours: {result['total_hours']}")
for step in result["steps"]:
    deps = f" (depends on: {step['dependencies']})" if step["dependencies"] else ""
    print(f"  {step['id']}. [{step['channel']}] {step['action']}{deps} — {step['estimated_hours']}h")

## Pattern 4: Multi-Agent Collaboration

Multiple specialised agents work together, each with distinct expertise.
In MarTech: a researcher, strategist, and copywriter collaborate on a campaign.

In [ ]:
AGENTS = {
    "researcher": "You are a market research analyst. Analyse the target audience and provide data-driven insights.",
    "strategist": "You are a campaign strategist. Given research insights, create a channel strategy with budget allocation.",
    "copywriter": "You are a conversion copywriter. Given the strategy, write email subject lines and ad headlines."
}

def multi_agent_pattern(brief: str) -> dict:
    """Run a sequential multi-agent pipeline on a campaign brief."""
    outputs = {}

    if USE_MOCK:
        outputs["researcher"] = "Target: B2B SaaS, 50-500 employees. Key pain: manual reporting. Buying trigger: quarterly planning."
        outputs["strategist"] = "Channels: LinkedIn (40%), Google Search (35%), Email (25%). Budget: $15K/month. KPI: MQLs."
        outputs["copywriter"] = 'Headlines: "Stop Manual Reports — Automate in 5 Minutes" | "Your Data Team Will Thank You"'
    else:
        context = brief
        for role, system_prompt in AGENTS.items():
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Brief: {brief}\n\nPrevious context: {context}"}
            ]
            outputs[role] = chat(messages)
            context = outputs[role]

    return outputs

# Demo
results = multi_agent_pattern("Launch campaign for AI-powered analytics platform targeting mid-market SaaS companies")
for agent, output in results.items():
    print(f"\n🔹 {agent.upper()}:\n{output}")

## Pattern 5: Memory

The agent retains and leverages conversation history for context-aware responses.
In MarTech: remember past interactions with a prospect for personalised follow-ups.

In [ ]:
class MemoryAgent:
    """Agent with conversation memory for personalised marketing interactions."""

    def __init__(self):
        self.conversation_history: list[dict] = [
            {"role": "system", "content": (
                "You are a customer success manager. Remember all previous interactions "
                "and use that context to personalise every response. Reference specific "
                "details the customer has shared before."
            )}
        ]

    def chat(self, user_message: str) -> str:
        """Send a message with full conversation history."""
        self.conversation_history.append({"role": "user", "content": user_message})

        if USE_MOCK:
            turn = len([m for m in self.conversation_history if m["role"] == "user"])
            mock_responses = [
                "Welcome! I see you're interested in our analytics platform. What's your main use case?",
                "Great — since you mentioned dashboard reporting for your 50-person team, I'd recommend our Team plan.",
                "Absolutely! Given your earlier interest in automated reporting and your team size, I'll set up a demo focused on those features."
            ]
            response = mock_responses[min(turn - 1, len(mock_responses) - 1)]
        else:
            response = chat(self.conversation_history)

        self.conversation_history.append({"role": "assistant", "content": response})
        return response

# Demo: Multi-turn conversation with memory
agent = MemoryAgent()
conversations = [
    "Hi, I'm looking at your analytics platform.",
    "We need dashboard reporting for about 50 people.",
    "Can you set up a demo for our team?"
]
for msg in conversations:
    print(f"\nUser: {msg}")
    print(f"Agent: {agent.chat(msg)}")

print(f"\n--- Memory size: {len(agent.conversation_history)} messages ---")

## Pattern 6: Guardrails

Safety checks that validate agent outputs before they reach the user or downstream systems.
In MarTech: ensure all generated content meets brand guidelines and compliance requirements.

In [ ]:
BRAND_RULES = {
    "banned_words": ["cheap", "buy now", "act fast", "limited time", "guaranteed"],
    "max_exclamation_marks": 1,
    "require_cta": True,
    "max_length": 500
}

def brand_safety_guardrail(content: str) -> dict:
    """Check content against brand guidelines. Returns pass/fail with issues."""
    issues = []

    # Check banned words
    for word in BRAND_RULES["banned_words"]:
        if word.lower() in content.lower():
            issues.append(f"Contains banned phrase: '{word}'")

    # Check exclamation marks
    if content.count("!") > BRAND_RULES["max_exclamation_marks"]:
        issues.append(f"Too many exclamation marks ({content.count('!')}, max {BRAND_RULES['max_exclamation_marks']})")

    # Check length
    if len(content) > BRAND_RULES["max_length"]:
        issues.append(f"Content too long ({len(content)} chars, max {BRAND_RULES['max_length']})")

    return {"passed": len(issues) == 0, "issues": issues, "content": content}

def guardrailed_generation(prompt: str) -> dict:
    """Generate content with brand safety guardrail."""
    if USE_MOCK:
        content = "Buy now! Act fast! This guaranteed cheap deal won't last!!!"
    else:
        content = chat([{"role": "user", "content": prompt}])

    check = brand_safety_guardrail(content)
    if not check["passed"]:
        print(f"GUARDRAIL BLOCKED — Issues: {check['issues']}")
        # In production: re-generate with stricter prompt
        content = "Discover how our platform transforms your marketing analytics — schedule a demo today."
        check = brand_safety_guardrail(content)
        print(f"Re-generated content passes: {check['passed']}")

    return check

# Demo
result = guardrailed_generation("Write a promotional email for our SaaS platform")
print(f"\nFinal content: {result['content']}")
print(f"Passed: {result['passed']}")

## Key Takeaways

| Pattern | MarTech Use Case | When to Use |
|---------|-----------------|-------------|
| **Reflection** | Ad copy optimisation | When output quality matters and iteration is cheap |
| **Tool Use** | CRM/CDP data access | When the agent needs real-time external data |
| **Planning** | Campaign orchestration | When tasks have dependencies and sequencing matters |
| **Multi-Agent** | Full campaign creation | When different expertise is needed at each stage |
| **Memory** | Customer conversations | When personalisation depends on interaction history |
| **Guardrails** | Brand safety | When outputs must meet compliance or brand standards |

These patterns are the building blocks for every project in this book.
Projects 1–2 combine Tool Use + Guardrails, Projects 3–4 use Multi-Agent + Planning,
and the capstone (Project 8) uses all six patterns together.

**Next:** See these patterns in action → Chapter 3 (OpenAI Agents SDK)